# LLMBackend Usage

`LLMBackend` implements krrood's `GenerativeBackend`: when a `Match` contains `...`, the backend fills those free slots from the LLM and the serialized `SymbolGraph` world context.

Current usage patterns:
1. **`instance_from_match()`** — action type is known; return a concrete action instance without creating a plan node.
2. **`plan_from_match()`** — action type is known; return a PyCRAM plan node for the resolved match.
3. **Context-level `LLMBackend`** — install the backend on a context and use `execute_single()`.
4. **Mid-session swap** — mix a normal krrood backend and `LLMBackend` in the same session.


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
symbol_type = Body  # optional — defaults to Symbol inside llmr


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [2]:
from dotenv import load_dotenv
load_dotenv("../.env")
from llmr.reasoning.llm_provider import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini")

---
## 1 — Known Action Type: `instance_from_match()` and `plan_from_match()`

Use `instance_from_match()` when you only want the resolved action instance. Use `plan_from_match()` when you want a `PlanNode` that can later be performed.


In [3]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.datastructures.grasp import GraspDescription

from llmr import plan_from_match, instance_from_match

INSTRUCTION = "pick up the milk from the table"

def pick_up_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

param_match = pick_up_match()

action_instance = instance_from_match(
    match=param_match,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("Resolved action type:", type(action_instance).__name__)
print(action_instance)


Resolved action type: PickUpAction
PickUpAction(object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('44841e53-996b-4890-9da8-78a474e9ffa0'), index=219), arm=RIGHT, grasp_description=GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('7e1f985a-2b47-42b3-b255-61a67ecacfbc'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('1e1c54d1-56be-49a6-974e-7a6df1cd1178'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('45759a5e-ad8f-4a90-9d84-6771013e3316'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('a896252e-167b-4255-a45f-45586d048978'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('eec179b1-31ce-4097-b2af-90642589c1c1'), index=22), manipulator=None, sensors=[Camera(name=

In [ ]:
plan_match = pick_up_match()

plan_node = plan_from_match(
    match=plan_match,
    context=context,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("Backend type:", type(context.query_backend).__name__)
print("Plan node   :", plan_node)


In [ ]:
action_instance


In [ ]:
plan_node.__dict__


In [ ]:
# Run only in a live simulation environment.
# plan_node.perform()


In [ ]:
# Inspect the generated plan graph after execution, if available.
# plan_node.plan.plan_graph.nodes()


---
## 2 — Context-level

Pass `LLMBackend` at context construction. Every plan in this context goes
through the LLM — no per-plan wiring needed.  Update `instruction` per-plan.


In [ ]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.datastructures.dataclasses import Context
from pycram.datastructures.grasp import GraspDescription

from llmr.backend import LLMBackend
from llmr.pycram import execute_single

llm_context = Context.from_world(
    world,
    query_backend=LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction="go to the kitchen table",
        strict_required=True,
    ),
)

print("Backend type:", type(llm_context.query_backend).__name__)


In [ ]:
# Plan 1 — uses the backend set on llm_context.
nav_match = Match(NavigateAction)(target_location=...)
nav_node = execute_single(nav_match, llm_context)

# Run only in a live simulation environment.
# nav_node.perform()


In [ ]:
# Plan 2 — update instruction for the next action, same context.
llm_context.query_backend.instruction = "pick up the milk from the table"

pick_match = Match(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=Match(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        manipulator=...,
    ),
)
pick_node = execute_single(pick_match, llm_context)

# Run only in a live simulation environment.
# pick_node.perform()


---
## 3 — Mid-session swap

Start with `ProbabilisticBackend` for routine actions. Switch to `LLMBackend`
via `plan_from_match()` for novel underspecified actions, then restore.


In [ ]:
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core import PickUpAction, PlaceAction
from pycram.datastructures.grasp import GraspDescription

from llmr.pycram import execute_single
from llmr.backend import LLMBackend

# Session starts with probabilistic backend.
prob_backend = ProbabilisticBackend()
context.query_backend = prob_backend
print("Start backend:", type(context.query_backend).__name__)


In [ ]:
# Routine pick — handled by the currently installed backend.
pick_match = Match(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=Match(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        manipulator=...,
    ),
)
pick_node = execute_single(pick_match, context)

# Run only in a live simulation environment.
# pick_node.perform()


In [ ]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core import PlaceAction

from llmr import plan_from_match

# Novel action — use plan_from_match() for this one underspecified action.
place_match = Match(PlaceAction)(object_designator=..., target_location=...)

place_node = plan_from_match(
    match=place_match,
    context=context,
    llm=llm,
    symbol_type=symbol_type,
    instruction="put the milk gently inside the fridge on the top shelf",
    strict_required=True,
)
print("Swapped backend:", type(context.query_backend).__name__)

# Run only in a live simulation environment.
# place_node.perform()

context.query_backend = prob_backend
print("Restored backend:", type(context.query_backend).__name__)


In [ ]:
# Restore probabilistic backend for remaining session.
context.query_backend = prob_backend
print("Restored backend:", type(context.query_backend).__name__)
